Step2 데이터 치환하기

In [86]:
!pip -q install lxml xmltodict 

In [87]:
from pathlib import Path
from zipfile import ZipFile, ZIP_DEFLATED, ZIP_STORED
from shutil import copytree, rmtree
import tempfile
import json
import copy
import xml.etree.ElementTree as ET


def extract_hwpx(hwpx_path: str | Path, output_dir: str | Path) -> Path:
    """HWPX 압축 해제"""

    hwpx_path = Path(hwpx_path)
    output_dir = Path(output_dir)

    if not hwpx_path.exists():
        raise FileNotFoundError(f"HWPX 파일이 없습니다: {hwpx_path}")

    if output_dir.exists():
        rmtree(output_dir)

    output_dir.mkdir(parents=True, exist_ok=True)

    with ZipFile(hwpx_path, "r") as zip_file:
        zip_file.extractall(output_dir)

    return output_dir


def hwpx_xml_to_json(xml_dir: str | Path, save_path: str | Path) -> dict:
    """section0.xml, header.xml, settings.xml -> JSON"""

    xml_dir = Path(xml_dir)
    save_path = Path(save_path)

    targets = {
        "section": xml_dir / "Contents" / "section0.xml",
        "header": xml_dir / "Contents" / "header.xml",
        "settings": xml_dir / "settings.xml",
    }

    def xml_to_dict(elem):
        return {
            "tag": elem.tag,  # 네임스페이스 보존
            "attrs": dict(elem.attrib),
            "text": elem.text or "",
            "tail": elem.tail or "",
            "children": [xml_to_dict(child) for child in elem],
        }

    result = {}

    for key, path in targets.items():
        if not path.exists():
            raise FileNotFoundError(f"필수 XML이 없습니다: {path}")

        tree = ET.parse(path)
        root = tree.getroot()

        result[key] = {
            "source_path": str(path),
            "root_tag": root.tag,
            "data": xml_to_dict(root),
        }

    save_path.parent.mkdir(parents=True, exist_ok=True)

    with open(save_path, "w", encoding="utf-8") as f:
        json.dump(result, f, ensure_ascii=False, indent=2)

    print(f"JSON 저장 완료: {save_path}")
    return result


def local_tag(tag: str) -> str:
    """{namespace}tag -> tag"""

    return tag.split("}", 1)[-1] if "}" in tag else tag


def make_render_json(file_json_path: str | Path, save_path: str | Path) -> dict:
    """원본 3계층 보존 + 렌더링용 render 필드 추가"""

    file_json_path = Path(file_json_path)
    save_path = Path(save_path)

    with open(file_json_path, "r", encoding="utf-8") as f:
        file_json = json.load(f)

    def walk(node, tag=None):
        result = []

        if tag is None or local_tag(node.get("tag", "")) == tag:
            result.append(node)

        for child in node.get("children", []):
            result.extend(walk(child, tag))

        return result

    def first_child(node, tag):
        for child in node.get("children", []):
            if local_tag(child.get("tag", "")) == tag:
                return child
        return None

    def to_int(value, default=None):
        try:
            return int(value)
        except (TypeError, ValueError):
            return default

    def get_text_from_run(run):
        texts = []

        for child in run.get("children", []):
            if local_tag(child.get("tag", "")) == "t":
                texts.append(child.get("text", ""))

        return "".join(texts)

    header_root = file_json["header"]["data"]
    section_root = file_json["section"]["data"]
    settings_root = file_json["settings"]["data"]

    font_map = {}

    for font in walk(header_root, "font"):
        attrs = font.get("attrs", {})
        font_id = attrs.get("id")
        face = attrs.get("face")

        if font_id is not None and face:
            font_map[font_id] = face

    char_style_map = {}

    for char_pr in walk(header_root, "charPr"):
        attrs = char_pr.get("attrs", {})
        char_id = attrs.get("id")

        font_ref = first_child(char_pr, "fontRef")
        font_id = None

        if font_ref:
            font_id = font_ref.get("attrs", {}).get("hangul")

        italic_node = first_child(char_pr, "italic")
        bold_node = first_child(char_pr, "bold")
        underline_node = first_child(char_pr, "underline")
        strikeout_node = first_child(char_pr, "strikeout")

        height = to_int(attrs.get("height"))

        char_style_map[char_id] = {
            "id": char_id,
            "font_id": font_id,
            "font": font_map.get(font_id),
            "height": height,
            "size_px_guess": height // 100 if height is not None else None,
            "textColor": attrs.get("textColor"),
            "shadeColor": attrs.get("shadeColor"),
            "bold": bold_node is not None,
            "italic": italic_node is not None,
            "underline": underline_node.get("attrs", {}) if underline_node else None,
            "strikeout": strikeout_node.get("attrs", {}) if strikeout_node else None,
            "raw_attrs": attrs,
        }

    para_style_map = {}

    for para_pr in walk(header_root, "paraPr"):
        attrs = para_pr.get("attrs", {})
        para_id = attrs.get("id")

        align = None
        heading = None
        line_spacing = None

        for child in para_pr.get("children", []):
            tag = local_tag(child.get("tag", ""))
            cattrs = child.get("attrs", {})

            if tag == "align":
                align = cattrs
            elif tag == "heading":
                heading = cattrs
            elif tag == "lineSpacing":
                line_spacing = cattrs

        para_style_map[para_id] = {
            "id": para_id,
            "align": align,
            "heading": heading,
            "lineSpacing": line_spacing,
            "raw_attrs": attrs,
        }

    paragraphs = []

    for p_index, p in enumerate(section_root.get("children", [])):
        if local_tag(p.get("tag", "")) != "p":
            continue

        p_attrs = p.get("attrs", {})
        para_id = p_attrs.get("paraPrIDRef")

        runs = []
        layout = None

        for run_index, child in enumerate(p.get("children", [])):
            child_tag = local_tag(child.get("tag", ""))

            if child_tag == "run":
                run_attrs = child.get("attrs", {})
                char_id = run_attrs.get("charPrIDRef")
                text = get_text_from_run(child)

                runs.append({
                    "type": "text_run",
                    "run_index": run_index,
                    "text": text,
                    "charPrIDRef": char_id,
                    "style": char_style_map.get(char_id),
                    "raw_attrs": run_attrs,
                })

            elif child_tag == "linesegarray":
                linesegs = []

                for lineseg in child.get("children", []):
                    if local_tag(lineseg.get("tag", "")) == "lineseg":
                        a = lineseg.get("attrs", {})

                        linesegs.append({
                            "textpos": to_int(a.get("textpos")),
                            "vertpos": to_int(a.get("vertpos")),
                            "vertsize": to_int(a.get("vertsize")),
                            "textheight": to_int(a.get("textheight")),
                            "baseline": to_int(a.get("baseline")),
                            "spacing": to_int(a.get("spacing")),
                            "horzpos": to_int(a.get("horzpos")),
                            "horzsize": to_int(a.get("horzsize")),
                            "flags": to_int(a.get("flags")),
                            "raw_attrs": a,
                        })

                layout = {
                    "linesegs": linesegs
                }

        paragraphs.append({
            "type": "paragraph",
            "index": p_index,
            "paraPrIDRef": para_id,
            "styleIDRef": p_attrs.get("styleIDRef"),
            "paragraph_style": para_style_map.get(para_id),
            "runs": runs,
            "layout": layout,
            "raw_attrs": p_attrs,
        })

    result_json = copy.deepcopy(file_json)

    result_json["render"] = {
        "type": "hwpx_render_json",
        "source": {
            "section": file_json["section"].get("source_path"),
            "header": file_json["header"].get("source_path"),
            "settings": file_json["settings"].get("source_path"),
        },
        "maps": {
            "fonts": font_map,
            "char_styles": char_style_map,
            "para_styles": para_style_map,
        },
        "document": {
            "paragraphs": paragraphs,
            "settings_raw": settings_root,
        },
    }

    save_path.parent.mkdir(parents=True, exist_ok=True)

    with open(save_path, "w", encoding="utf-8") as f:
        json.dump(result_json, f, ensure_ascii=False, indent=2)

    print(f"렌더 JSON 저장 완료: {save_path}")
    return result_json


def json_node_to_xml_elem(data):
    """JSON node -> XML element"""

    elem = ET.Element(
        str(data["tag"]),
        {
            str(k): str(v)
            for k, v in data.get("attrs", {}).items()
            if v is not None
        }
    )

    elem.text = data.get("text") or None
    elem.tail = data.get("tail") or None

    for child in data.get("children", []):
        elem.append(json_node_to_xml_elem(child))

    return elem


def write_json_xml_parts(json_data: dict, output_xml_dir: str | Path) -> Path:
    """JSON의 section/header/settings를 XML 파일로 저장"""

    output_xml_dir = Path(output_xml_dir)

    targets = {
        "section": output_xml_dir / "Contents" / "section0.xml",
        "header": output_xml_dir / "Contents" / "header.xml",
        "settings": output_xml_dir / "settings.xml",
    }

    for key, path in targets.items():
        if key not in json_data:
            raise KeyError(f"json_data에 '{key}'가 없습니다.")

        root_elem = json_node_to_xml_elem(json_data[key]["data"])

        path.parent.mkdir(parents=True, exist_ok=True)

        tree = ET.ElementTree(root_elem)
        ET.indent(tree, space="  ", level=0)
        tree.write(path, encoding="utf-8", xml_declaration=True)

        print(f"XML 저장 완료: {path}")

    return output_xml_dir


def zip_hwpx_dir(source_dir: str | Path, output_hwpx: str | Path) -> Path:
    """HWPX 디렉터리 압축"""

    source_dir = Path(source_dir)
    output_hwpx = Path(output_hwpx)

    output_hwpx.parent.mkdir(parents=True, exist_ok=True)

    with ZipFile(output_hwpx, "w") as z:
        mimetype = source_dir / "mimetype"

        if mimetype.exists():
            z.write(
                mimetype,
                "mimetype",
                compress_type=ZIP_STORED
            )

        for file in source_dir.rglob("*"):
            if not file.is_file():
                continue

            rel = file.relative_to(source_dir)

            if str(rel).replace("\\", "/") == "mimetype":
                continue

            z.write(
                file,
                rel,
                compress_type=ZIP_DEFLATED
            )

    return output_hwpx


def json_to_hwpx(json_data: dict, template_dir: str | Path, output_hwpx: str | Path) -> Path:
    """JSON을 XML로 변환 후 템플릿 복사본에 치환하여 HWPX 생성"""

    template_dir = Path(template_dir)
    output_hwpx = Path(output_hwpx)

    if not template_dir.exists():
        raise FileNotFoundError(f"템플릿 폴더가 없습니다: {template_dir}")

    with tempfile.TemporaryDirectory() as temp_dir:
        work_dir = Path(temp_dir) / "hwpx"
        copytree(template_dir, work_dir)

        write_json_xml_parts(json_data, work_dir)
        zip_hwpx_dir(work_dir, output_hwpx)

    print(f"HWPX 생성 완료: {output_hwpx}")
    return output_hwpx


def xml_to_hwpx(xml_dir: str | Path, template_dir: str | Path, output_hwpx: str | Path) -> Path:
    """추출된 XML 폴더를 템플릿 복사본에 치환하여 HWPX 생성"""

    from shutil import copy2

    xml_dir = Path(xml_dir)
    template_dir = Path(template_dir)
    output_hwpx = Path(output_hwpx)

    if not xml_dir.exists():
        raise FileNotFoundError(f"XML 폴더가 없습니다: {xml_dir}")

    if not template_dir.exists():
        raise FileNotFoundError(f"템플릿 폴더가 없습니다: {template_dir}")

    replace_files = [
        Path("Contents") / "section0.xml",
        Path("Contents") / "header.xml",
        Path("settings.xml"),
    ]

    with tempfile.TemporaryDirectory() as temp_dir:
        work_dir = Path(temp_dir) / "hwpx"
        copytree(template_dir, work_dir)

        for rel_path in replace_files:
            src = xml_dir / rel_path
            dst = work_dir / rel_path

            if not src.exists():
                raise FileNotFoundError(f"치환할 XML이 없습니다: {src}")

            dst.parent.mkdir(parents=True, exist_ok=True)
            copy2(src, dst)

            print(f"XML 치환 완료: {rel_path}")

        zip_hwpx_dir(work_dir, output_hwpx)

    print(f"HWPX 생성 완료: {output_hwpx}")
    return output_hwpx

In [88]:
from pathlib import Path

input_file_path = Path("HWPX 파일") / "ex_글자.hwpx"
output_dir = Path("step2_output")
xml_dir = output_dir / "xml"

json_file_path = output_dir / f"{input_file_path.stem}.json"
render_json_path = output_dir / f"{input_file_path.stem}_render.json"
output_hwpx = output_dir / "result.hwpx"

template_dir = Path("빈파일_xml")

# 1. HWPX -> XML
extract_hwpx(input_file_path, xml_dir)

# 2. XML -> 원본 보존 JSON
file_json = hwpx_xml_to_json(
    xml_dir=xml_dir,
    save_path=json_file_path
)

# 3. 원본 보존 JSON -> render 필드 추가 JSON
render_json = make_render_json(
    file_json_path=json_file_path,
    save_path=render_json_path
)

# 4. JSON -> HWPX
json_to_hwpx(
    json_data=render_json,
    template_dir=template_dir,
    output_hwpx=output_hwpx
)

JSON 저장 완료: step2_output\ex_글자.json
렌더 JSON 저장 완료: step2_output\ex_글자_render.json
XML 저장 완료: C:\Users\hyun\AppData\Local\Temp\tmpzvifhkf7\hwpx\Contents\section0.xml
XML 저장 완료: C:\Users\hyun\AppData\Local\Temp\tmpzvifhkf7\hwpx\Contents\header.xml
XML 저장 완료: C:\Users\hyun\AppData\Local\Temp\tmpzvifhkf7\hwpx\settings.xml
HWPX 생성 완료: step2_output\result.hwpx


WindowsPath('step2_output/result.hwpx')

In [89]:
render_json["section"]["data"]["children"][1]["children"][0]["children"][0]["text"] = "치환 테스트"

json_to_hwpx(
    json_data=render_json,
    template_dir=template_dir,
    output_hwpx=output_dir / "result_replace.hwpx"
)

XML 저장 완료: C:\Users\hyun\AppData\Local\Temp\tmpbikjte9u\hwpx\Contents\section0.xml
XML 저장 완료: C:\Users\hyun\AppData\Local\Temp\tmpbikjte9u\hwpx\Contents\header.xml
XML 저장 완료: C:\Users\hyun\AppData\Local\Temp\tmpbikjte9u\hwpx\settings.xml
HWPX 생성 완료: step2_output\result_replace.hwpx


WindowsPath('step2_output/result_replace.hwpx')